## Colab setup
Verify GPU, mount Drive, clone the repo, install requirements, and create the project tree on Drive.


In [ ]:
# 1. GPU check
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# 2. Mount Drive (skipped automatically when not on Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/cs4782_patchtst_project'
    IN_COLAB = True
except Exception:
    PROJECT_ROOT = '.'
    IN_COLAB = False
print('PROJECT_ROOT =', PROJECT_ROOT, ' IN_COLAB =', IN_COLAB)


In [ ]:
# 3. Clone repo (Colab only). Update REPO_URL in scripts/build_notebooks.py
# and re-run that script to regenerate notebooks if the URL changes.
REPO_URL = 'https://github.com/Ash1R/ATSW64W-experiments.git'
if IN_COLAB:
    import os, subprocess
    os.chdir('/content')
    # Derive the clone directory from the URL's basename so it matches the repo name.
    REPO_DIRNAME = REPO_URL.rstrip('/').rsplit('/', 1)[-1]
    if REPO_DIRNAME.endswith('.git'):
        REPO_DIRNAME = REPO_DIRNAME[:-4]
    if not os.path.isdir(f'/content/{REPO_DIRNAME}'):
        # check=True so a bad URL fails loudly here instead of crashing the next chdir.
        subprocess.run(['git', 'clone', REPO_URL, REPO_DIRNAME], check=True)
    os.chdir(f'/content/{REPO_DIRNAME}')
    subprocess.run(['git', 'pull'], check=False)
print('cwd =', __import__('os').getcwd())


In [ ]:
# 4. Install requirements (best-effort; resolved relative to the repo root
# regardless of where the kernel started, so headless `nbconvert` runs work).
import subprocess, sys, os
_req_dir = os.getcwd()
for _ in range(4):
    if os.path.isfile(os.path.join(_req_dir, 'requirements.txt')):
        break
    _req_dir = os.path.dirname(_req_dir)
_req = os.path.join(_req_dir, 'requirements.txt')
if os.path.isfile(_req):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', _req], check=False)
else:
    print('skipping pip install — requirements.txt not found from', os.getcwd())


In [ ]:
# 5. Make the project's `code/` directory importable.
# We add `code/` itself to sys.path (not the repo root) because the
# stdlib already ships a module named `code` that the IPython kernel
# imports before this cell runs — shadowing that cleanly is messy.
# This way every import is `from utils...`, `from data...`, `from models...`.
import sys, os
# When run with `jupyter nbconvert --execute`, the kernel's cwd is
# the notebook's directory (`notebooks/`), so locate the repo root
# by walking up until we find `code/`. On Colab we already chdir'd
# into the cloned repo above.
REPO_DIR = os.getcwd()
for _ in range(4):
    if os.path.isdir(os.path.join(REPO_DIR, 'code')):
        break
    REPO_DIR = os.path.dirname(REPO_DIR)
CODE_DIR = os.path.join(REPO_DIR, 'code')
if not os.path.isdir(CODE_DIR):
    raise RuntimeError(f'could not locate code/ from {os.getcwd()}')
os.chdir(REPO_DIR)
for p in (REPO_DIR, CODE_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)
print('REPO_DIR =', REPO_DIR)
from utils.colab import ensure_dirs
subdirs = ensure_dirs(PROJECT_ROOT)
for k, v in subdirs.items():
    print(f'{k:>12}  {v}')


## Beyond-the-paper extensions
Two experiments not in the PatchTST paper:

  1. **Patch order shuffling** (inference-only, ~1 min)
  2. **Channel grouping spectrum** (trains 4 models, ~60-90 min on a T4)

**This notebook is fully self-contained** — both experiment implementations are defined inline below, so it works whether or not the GitHub clone has the latest `code/extensions/` files. Just run all cells top-to-bottom. The notebook writes the raw extension JSON files and a complete `nb06_extension_results_manifest.json` to `<project_root>/results/extensions/`, then renders the two final figures inline and saves them to `<project_root>/results/figures/`. On Colab, `<project_root>` is on Google Drive, so these outputs persist after the runtime stops.


### A. Setup and load the headline PatchTST checkpoint
Imports, paths, and the loaded `weather_patchtst_L336_T96_P16S8_seed42` model.


In [ ]:
# Imports — use top-level package names because SETUP_CELLS added
# `code/` directly to sys.path (the stdlib already owns `code`).
import os, json
import numpy as np, torch
import torch.nn as nn
import matplotlib.pyplot as plt
from data.dataset import build_data_bundle
from data.preprocessing import StandardScaler
from models import build_model
from models.patchtst import _num_patches
from utils.device import get_device
from train import train_from_config

EXT_DIR = os.path.join(PROJECT_ROOT, 'results', 'extensions')
FIG_DIR = os.path.join(PROJECT_ROOT, 'results', 'figures')
os.makedirs(EXT_DIR, exist_ok=True); os.makedirs(FIG_DIR, exist_ok=True)
device = get_device()
print('device:', device)


In [ ]:
CKPT_NAME = 'weather_patchtst_L336_T96_P16S8_seed42'
ckpt_path = os.path.join(PROJECT_ROOT, 'results', 'checkpoints', CKPT_NAME + '.pt')
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
cfg = ckpt['config']
bundle = build_data_bundle(
    project_root=PROJECT_ROOT, dataset=cfg['dataset'],
    seq_len=cfg['seq_len'], pred_len=cfg['pred_len']
)
bundle.scaler = StandardScaler(
    mean=np.asarray(ckpt['scaler_mean']), std=np.asarray(ckpt['scaler_std'])
)
_, _, test_loader = bundle.loaders(batch_size=64, shuffle_train=False)
model = build_model(cfg['model'],
    seq_len=cfg['seq_len'], pred_len=cfg['pred_len'], num_channels=bundle.num_channels,
    patch_len=cfg.get('patch_len', 16), stride=cfg.get('stride', 8),
    d_model=cfg.get('d_model', 128), n_heads=cfg.get('n_heads', 16),
    num_layers=cfg.get('num_layers', 3), d_ff=cfg.get('d_ff', 256),
    dropout=cfg.get('dropout', 0.2)
).to(device)
model.load_state_dict(ckpt['model'])
model.eval()
print(f'loaded {CKPT_NAME} ({sum(p.numel() for p in model.parameters()):,} params)')


### B. Patch-content shuffling before positional embeddings
Permute a random fraction of patch **contents** at inference; sweep frac in {0, 0.1, 0.25, 0.5, 0.75, 1.0}.

**Beyond the paper:** PatchTST never tests whether patch contents must appear in the correct temporal slots. We shuffle patch contents **before** adding positional embeddings, so positional slots stay fixed while the wrong local subseries appears at those slots. If MSE rises, PatchTST depends on temporal patch placement, not just unordered local patch statistics.


In [ ]:
def _forward_with_pre_pos_content_shuffle(model, x, frac, rng):
    B, L, M = x.shape
    x_t = x.transpose(1, 2)
    x_norm, mean, std = model._instance_norm(x_t)
    patches = model.embed(x_norm)  # [B, M, N, d_model]
    N = patches.shape[2]
    if frac > 0:
        n_shuf = max(2, int(round(frac * N)))
        idx = np.arange(N)
        pos = np.sort(rng.choice(N, size=n_shuf, replace=False))
        perm = idx.copy()
        perm[pos] = rng.permutation(perm[pos])
        perm_t = torch.as_tensor(perm, device=patches.device, dtype=torch.long)
        patches = patches.index_select(2, perm_t)
    tokens = patches.reshape(B * M, N, model.d_model)
    tokens = tokens + model.pos_embed  # positional slots stay fixed
    tokens = model.encoder_dropout(tokens)
    encoded = model.encoder(tokens)
    flat = encoded.reshape(B * M, N * model.d_model)
    pred = model.head(flat).reshape(B, M, model.pred_len)
    pred = pred * std + mean
    return pred.transpose(1, 2)

def patch_shuffle_curve(model, loader, device, fractions, seed=0):
    N = model.num_patches
    out = {'type': 'pre_positional_patch_content_shuffle',
           'fractions': list(fractions), 'mse': [], 'mae': []}
    model.eval()
    for frac in out['fractions']:
        rng = np.random.RandomState(seed + int(frac * 1000))
        preds, trues = [], []
        with torch.no_grad():
            for x, y in loader:
                preds.append(_forward_with_pre_pos_content_shuffle(
                    model, x.to(device), frac, rng).cpu().numpy())
                trues.append(y.numpy())
        p = np.concatenate(preds, 0); t = np.concatenate(trues, 0)
        out['mse'].append(float(((p - t) ** 2).mean()))
        out['mae'].append(float(np.abs(p - t).mean()))
    return out

ps_out = patch_shuffle_curve(model, test_loader, device,
    fractions=(0.0, 0.1, 0.25, 0.5, 0.75, 1.0), seed=0)
ps_out['baseline_mse'] = ps_out['mse'][0]
ps_out['checkpoint'] = CKPT_NAME
with open(os.path.join(EXT_DIR, 'patch_shuffle.json'), 'w') as f:
    json.dump(ps_out, f, indent=2)
for f, m in zip(ps_out['fractions'], ps_out['mse']):
    print(f'  shuffled {f*100:5.1f}%  MSE = {m:.4f}')


### B2. No positional embedding ablation
Evaluate the same trained checkpoint with `pos_embed` zeroed at inference time. Patch contents remain in the correct order, so this isolates the contribution of the learned positional embedding itself.


In [13]:
def evaluate_no_positional_embedding(model, loader, device):
    model.eval()
    saved_pos = model.pos_embed.detach().clone()
    preds, trues = [], []
    try:
        model.pos_embed.data.zero_()
        with torch.no_grad():
            for x, y in loader:
                preds.append(model(x.to(device)).cpu().numpy())
                trues.append(y.numpy())
    finally:
        model.pos_embed.data.copy_(saved_pos)
    p = np.concatenate(preds, 0); t = np.concatenate(trues, 0)
    return {'type': 'zero_positional_embedding',
            'mse': float(((p - t) ** 2).mean()),
            'mae': float(np.abs(p - t).mean())}

nopos_out = evaluate_no_positional_embedding(model, test_loader, device)
nopos_out['baseline_mse'] = ps_out['baseline_mse']
nopos_out['checkpoint'] = CKPT_NAME
with open(os.path.join(EXT_DIR, 'no_positional_embedding.json'), 'w') as f:
    json.dump(nopos_out, f, indent=2)
pct = (nopos_out['mse'] - nopos_out['baseline_mse']) / nopos_out['baseline_mse'] * 100
print(f"no positional embedding  MSE = {nopos_out['mse']:.4f}  ({pct:+.1f}% vs baseline)")


no positional embedding  MSE = 0.1485  (+0.6% vs baseline)


### C. Channel grouping spectrum (k = 1, 3, 7, 21)
The paper presents the channel-axis treatment as binary (full-mix vs full-indep). We cluster Weather's 21 channels by Pearson correlation on the train split, then train one model per k. k=1 reduces to channel-mixing, k=21 to channel-independence.

**Beyond the paper:** if structured cross-channel mixing recovers signal that pure independence loses, that's a meaningful new design point. If the spectrum is monotone, the binary choice in the paper is empirically correct.

Implementation: model class + clustering helper are defined inline. We train each k with a manual training loop (Adam, lr 1e-4, 20 epochs, early-stop on val MSE) so the notebook doesn't need a model-registry change in `code/`.


In [ ]:
# 1. Channel-grouped PatchTST model (defined inline; no code/ changes required)
class ChannelGroupedPatchTST(nn.Module):
    def __init__(self, seq_len, pred_len, groups,
                 patch_len=16, stride=8, d_model=128, n_heads=16,
                 num_layers=3, d_ff=256, dropout=0.2):
        super().__init__()
        self.seq_len, self.pred_len = seq_len, pred_len
        self.patch_len, self.stride, self.d_model = patch_len, stride, d_model
        self.groups = [sorted(g) for g in groups]
        self.num_patches = _num_patches(seq_len, patch_len, stride)
        self.group_embed = nn.ModuleList(
            nn.Linear(patch_len * len(g), d_model) for g in self.groups)
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches, d_model))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads,
            dim_feedforward=d_ff, dropout=dropout, activation='gelu',
            batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.encoder_dropout = nn.Dropout(dropout)
        self.group_head = nn.ModuleList(
            nn.Linear(self.num_patches * d_model, pred_len * len(g)) for g in self.groups)
    @staticmethod
    def _instance_norm(x, eps=1e-5):
        mean = x.mean(-1, keepdim=True)
        std  = x.std(-1, keepdim=True, unbiased=False) + eps
        return (x - mean) / std, mean, std
    def _patchify(self, x_g):
        B, mg, L = x_g.shape
        pad = x_g[..., -1:].expand(B, mg, self.stride)
        xp = torch.cat([x_g, pad], dim=-1)
        p = xp.unfold(-1, self.patch_len, self.stride)
        return p.permute(0, 2, 1, 3).reshape(B, self.num_patches, mg * self.patch_len)
    def forward(self, x):
        B, L, M = x.shape
        x = x.transpose(1, 2)
        x_norm, mean, std = self._instance_norm(x)
        out = torch.zeros(B, M, self.pred_len, device=x.device, dtype=x.dtype)
        for g, embed, head in zip(self.groups, self.group_embed, self.group_head):
            idx = torch.as_tensor(g, device=x.device, dtype=torch.long)
            xg = x_norm.index_select(1, idx)
            patches = self._patchify(xg)
            tokens = embed(patches) + self.pos_embed
            tokens = self.encoder_dropout(tokens)
            enc = self.encoder(tokens)
            flat = enc.reshape(B, self.num_patches * self.d_model)
            pred_g = head(flat).reshape(B, len(g), self.pred_len)
            out.index_copy_(1, idx, pred_g)
        return (out * std + mean).transpose(1, 2)

def cluster_channels_by_correlation(train_data, k, seed=0):
    M = train_data.shape[1]
    if k <= 1: return [list(range(M))]
    if k >= M: return [[i] for i in range(M)]
    corr = np.corrcoef(train_data.T); np.fill_diagonal(corr, 0.0)
    score = np.abs(corr).mean(1)
    order = np.argsort(-score)
    rng = np.random.RandomState(seed); rng.shuffle(order)
    groups = [[] for _ in range(k)]
    for i, ch in enumerate(order):
        groups[i % k].append(int(ch))
    return [sorted(g) for g in groups]


In [ ]:
# 2. Manual train loop — same recipe as the rest of the project (Adam, lr 1e-4,
#    20 epochs, early-stop on val MSE). Doesn't depend on the build_model registry.
from utils.metrics import compute_metrics

def train_grouped_model(groups, k, epochs=20, lr=1e-4, patience=5):
    torch.manual_seed(42); np.random.seed(42)
    model_g = ChannelGroupedPatchTST(
        seq_len=cfg['seq_len'], pred_len=cfg['pred_len'], groups=groups,
        patch_len=cfg.get('patch_len', 16), stride=cfg.get('stride', 8),
        d_model=cfg.get('d_model', 128), n_heads=cfg.get('n_heads', 16),
        num_layers=cfg.get('num_layers', 3), d_ff=cfg.get('d_ff', 256),
        dropout=cfg.get('dropout', 0.2),
    ).to(device)
    opt = torch.optim.Adam(model_g.parameters(), lr=lr)
    train_loader, val_loader, _ = bundle.loaders(batch_size=32)
    best_val, best_state, bad = float('inf'), None, 0
    for ep in range(epochs):
        model_g.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(); loss = ((model_g(x) - y) ** 2).mean()
            loss.backward(); opt.step()
        model_g.eval()
        vals = []
        with torch.no_grad():
            for x, y in val_loader:
                vals.append(((model_g(x.to(device)) - y.to(device)) ** 2).mean().item())
        v = float(np.mean(vals))
        print(f'  k={k:>2} ep {ep+1:>2}/{epochs}  val_mse={v:.4f}')
        if v < best_val:
            best_val, best_state, bad = v, {kk: vv.detach().clone() for kk, vv in model_g.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience:
                print(f'  early-stop at epoch {ep+1}'); break
    model_g.load_state_dict(best_state)
    # test MSE on normalized scale
    model_g.eval(); preds, trues = [], []
    with torch.no_grad():
        for x, y in test_loader:
            preds.append(model_g(x.to(device)).cpu().numpy()); trues.append(y.numpy())
    p = np.concatenate(preds, 0); t = np.concatenate(trues, 0)
    return {'mse': float(((p - t) ** 2).mean()),
            'mae': float(np.abs(p - t).mean()),
            'params': sum(pp.numel() for pp in model_g.parameters())}


In [ ]:
train_data = bundle.train.data
cg_results = []
cg_groups = {}
for k in (1, 3, 7, 21):
    groups = cluster_channels_by_correlation(train_data, k=k, seed=0)
    cg_groups[str(k)] = groups
    print(f'\nk={k}: {len(groups)} groups, sizes = {[len(g) for g in groups]}')
    res = train_grouped_model(groups, k=k, epochs=20, lr=1e-4, patience=5)
    cg_results.append({'k': k, **res})
    print(f"  -> test MSE = {res['mse']:.4f}  params = {res['params']:,}")
cg_out = {'k_groups': [r['k'] for r in cg_results],
          'mse':      [r['mse'] for r in cg_results],
          'mae':      [r['mae'] for r in cg_results],
          'params':   [r['params'] for r in cg_results],
          'groups':   cg_groups}
with open(os.path.join(EXT_DIR, 'channel_grouping.json'), 'w') as f:
    json.dump(cg_out, f, indent=2)


### D. Render the two poster figures
Both figures are saved to `<project_root>/results/figures/` and shown inline.


In [ ]:
# Figure 1: pre-positional patch-content shuffle curve
fracs = np.array(ps_out['fractions'])
mse = np.array(ps_out['mse'])
fig, ax = plt.subplots(figsize=(4.4, 2.8), dpi=160)
ax.plot(fracs * 100, mse, 'o-', color='#b31b1b', lw=2.2, ms=7)
ax.axhline(mse[0], ls='--', color='#777', lw=1, label=f'unshuffled = {mse[0]:.3f}')
if 'nopos_out' in globals():
    ax.axhline(nopos_out['mse'], ls=':', color='#1f4fa6', lw=1.8,
               label=f'no pos embed = {nopos_out["mse"]:.3f}')
ax.set_xlabel('Patch contents shuffled before pos embed (%)')
ax.set_ylabel('Test MSE (normalized)')
ax.set_title('Does temporal patch placement matter?', fontweight='bold')
ax.grid(alpha=0.3); ax.legend(loc='upper left', fontsize=9)
fig.tight_layout()
out_png = os.path.join(FIG_DIR, 'ext_patch_shuffle.png')
fig.savefig(out_png, bbox_inches='tight', facecolor='white'); plt.show()
print('saved', out_png)


In [ ]:
# Figure 2: channel grouping spectrum (MSE vs k, with params on a twin axis)
ks = np.array(cg_out['k_groups'])
mses = np.array(cg_out['mse'])
params = np.array([p if p is not None else np.nan for p in cg_out['params']])
fig, ax1 = plt.subplots(figsize=(4.6, 2.8), dpi=160)
c_red, c_blue = '#b31b1b', '#1f4fa6'
ax1.plot(ks, mses, 'o-', color=c_red, lw=2.2, ms=8, label='Test MSE')
for k, m in zip(ks, mses):
    ax1.annotate(f'{m:.3f}', xy=(k, m), xytext=(0, 8),
                 textcoords='offset points', ha='center', fontsize=9, color=c_red)
ax1.set_xlabel('Channel groups k  (1 = full mixing  ->  21 = full independence)')
ax1.set_ylabel('Test MSE (normalized)', color=c_red)
ax1.tick_params(axis='y', labelcolor=c_red)
ax1.set_xscale('log'); ax1.set_xticks(ks); ax1.set_xticklabels([str(k) for k in ks])
ax1.grid(alpha=0.3)
ax2 = ax1.twinx()
ax2.plot(ks, params / 1e6, 's--', color=c_blue, lw=1.5, ms=6)
ax2.set_ylabel('Parameters (M)', color=c_blue)
ax2.tick_params(axis='y', labelcolor=c_blue)
ax1.set_title('Mixing  <->  independence spectrum', fontweight='bold')
fig.tight_layout()
out_png = os.path.join(FIG_DIR, 'ext_channel_grouping.png')
fig.savefig(out_png, bbox_inches='tight', facecolor='white'); plt.show()
print('saved', out_png)


### E. Push the complete nb6 results payload to Drive
This cell writes a single manifest JSON containing the actual extension numbers, channel groups, figure paths, and source checkpoint. In Colab this lands directly in Google Drive at `<project_root>/results/extensions/`.


In [ ]:
import datetime as _datetime_module

manifest = {
    'created_utc': _datetime_module.datetime.now(_datetime_module.timezone.utc).isoformat(),
    'project_root': os.path.abspath(PROJECT_ROOT),
    'in_colab': bool(IN_COLAB),
    'checkpoint': CKPT_NAME,
    'checkpoint_path': ckpt_path,
    'raw_result_files': {
        'patch_shuffle': os.path.join(EXT_DIR, 'patch_shuffle.json'),
        'channel_grouping': os.path.join(EXT_DIR, 'channel_grouping.json'),
        'no_positional_embedding': os.path.join(EXT_DIR, 'no_positional_embedding.json'),
    },
    'figure_files': {
        'patch_shuffle': os.path.join(FIG_DIR, 'ext_patch_shuffle.png'),
        'channel_grouping': os.path.join(FIG_DIR, 'ext_channel_grouping.png'),
    },
    'patch_shuffle': ps_out,
    'no_positional_embedding': nopos_out,
    'channel_grouping': cg_out,
}
manifest_path = os.path.join(EXT_DIR, 'nb06_extension_results_manifest.json')
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)

# Best-effort durability hint for Drive-backed paths before the runtime exits.
try:
    for path in [manifest_path, *manifest['raw_result_files'].values(), *manifest['figure_files'].values()]:
        if os.path.exists(path):
            with open(path, 'ab') as _f:
                _f.flush(); os.fsync(_f.fileno())
except Exception as e:
    print('Drive fsync skipped:', e)

print('nb6 results manifest saved to:', manifest_path)
for label, path in manifest['raw_result_files'].items():
    print(f'raw {label}:', path)
for label, path in manifest['figure_files'].items():
    print(f'figure {label}:', path)


### F. Summary
Both figures are now in `<project_root>/results/figures/` and the underlying numbers plus the complete nb6 manifest are in `<project_root>/results/extensions/`. Edit the poster's two extension-panel <img> tags to point at `ext_patch_shuffle.png` and `ext_channel_grouping.png`.


In [ ]:
print('\n=== Patch shuffling ===')
for f, m in zip(ps_out['fractions'], ps_out['mse']):
    pct = (m - ps_out['baseline_mse']) / ps_out['baseline_mse'] * 100
    print(f'  shuffled {f*100:5.1f}%  MSE = {m:.4f}  ({pct:+.1f}% vs unshuffled)')
print('\n=== Channel grouping ===')
for k, m, p in zip(cg_out['k_groups'], cg_out['mse'], cg_out['params']):
    label = ('full mixing' if k == 1 else 'full independence' if k == 21 else f'k={k}')
    print(f'  k={k:>2} ({label:<18}) MSE = {m:.4f}  params = {p:,}')
print('\nManifest:', manifest_path)
print('\nDone. Re-export the poster PDF to pick up the new figures.')
